# 02 - Chunking and Index Preparation

This notebook reads the cleaned document text, splits it into retrieval-friendly chunks,
writes the chunk table, and enables Change Data Feed for Vector Search.

In [0]:
import re
from typing import List

In [0]:
# -----------------------------
# Configuration
# -----------------------------
CATALOG = "workspace"
SCHEMA = "rag_demo"

PARSED_DOCS_TABLE = f"{CATALOG}.{SCHEMA}.parsed_docs_clean"
CHUNKS_TABLE = f"{CATALOG}.{SCHEMA}.rag_chunks"
DOC_TYPE = "runbook_library"

# -----------------------------
# Print
# -----------------------------
print(f"PARSED_DOCS_TABLE: {PARSED_DOCS_TABLE}")
print(f"CHUNKS_TABLE: {CHUNKS_TABLE}")
print(f"DOC_TYPE: {DOC_TYPE}")

## Helper Functions

In [0]:
def split_into_sections(clean_text: str) -> List[str]:
    """
    Split document into sections using numbered headings.
    Example: 1. Title, 2. Purpose
    """
    sections = re.split(r'\n(?=\d+\.\s)', clean_text)
    return [section.strip() for section in sections if section.strip()]


def build_chunk_df(file_path: str, file_name: str, sections: List[str]):
    """
    Build chunk DataFrame for Vector Search ingestion.
    """
    chunk_rows = [
        (f"chunk_{i}", file_path, file_name, DOC_TYPE, section)
        for i, section in enumerate(sections, start=1)
    ]

    return spark.createDataFrame(
        chunk_rows,
        ["chunk_id", "file_path", "file_name", "doc_type", "chunk"]
    )

## Read parsed document

In [0]:
parsed_df = spark.table(PARSED_DOCS_TABLE)
display(parsed_df)

## Split text into sections

In [0]:
row = parsed_df.collect()[0]
file_path = row["file_path"]
file_name = row["file_name"]
clean_text = row["text"]

sections = split_into_sections(clean_text)

print(f"Number of sections: {len(sections)}")
for section in sections[:3]:
    print("-" * 80)
    print(section[:500])

## Build and save chunk table

In [0]:
chunk_df = build_chunk_df(file_path, file_name, sections)
display(chunk_df)

chunk_df.write.mode("overwrite").saveAsTable(CHUNKS_TABLE)

## Enable Change Data Feed
Required for Delta Sync Vector Search index.

In [0]:
spark.sql(f"""
ALTER TABLE {CHUNKS_TABLE}
SET TBLPROPERTIES (delta.enableChangeDataFeed = true)
""")

spark.sql(f"SHOW TBLPROPERTIES {CHUNKS_TABLE}").show(truncate=False)